# 🔘🤖 Gradio LLM Chat

⚠️ **Remember to copy this notebook in your Drive and rename.**

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

## Setup

In [ ]:
import os
os.environ["TORCHINDUCTOR_DISABLE"] = "1"
os.environ["TORCHDYNAMO_DISABLE"] = "1"
os.environ["TORCH_COMPILER_DISABLE"] = "1"

import torch
torch._dynamo.disable()
torch.set_float32_matmul_precision('high')

In [ ]:
!pip install --upgrade transformers accelerate gradio --quiet

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

### *** Some models may require you to accept terms and conditions on HuggingFace

model_ids = ["Qwen/Qwen1.5-1.8B-Chat",
             "google/gemma-3-4b-it",
             "google/gemma-2b-it",
             "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
             "mistralai/Mistral-7B-Instruct-v0.3",
             "microsoft/phi-2"
            ]

sel_model = 1

# Load tokenizer and ensure pad token
model_id = model_ids[sel_model]
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token

# Load model with bfloat16 for Colab A100 / T4 / A10G
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

## Choose one chat function to run below
Reducing max_new_tokens will speed up chat responses

In [ ]:
# This chat function is designed for general-purpose language models not chat-tuned / instruction-tuned.

def chat(message, history):
    prompt = ""
    for turn in history:
        prompt += f"{turn['role']}: {turn['content']}\n"
    prompt += f"user: {message}\nassistant:"

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, padding=True).to(model.device)
    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    return reply

In [ ]:
# This chat function is intended for chat-tuned or instruction-following models that were trained with role-based or special token formats (e.g. Gemma, Qwen-Chat, Mistral).
# It uses <start_of_turn> and <end_of_turn> tokens to structure a multi-turn dialogue.

def chat(message, history):
    system_prompt = "<start_of_turn>system\nYou are a helpful assistant.<end_of_turn>\n"
    prompt = system_prompt

    for turn in history:
        role = turn["role"]
        content = turn["content"]
        if role == "user":
            prompt += f"<start_of_turn>user\n{content}<end_of_turn>\n"
        elif role == "assistant":
            prompt += f"<start_of_turn>model\n{content}<end_of_turn>\n"

    prompt += f"<start_of_turn>user\n{message}<end_of_turn>\n<start_of_turn>model\n"

    inputs = tokenizer(prompt, return_tensors="pt", padding=True, truncation=False)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    MAX_HISTORY_TOKENS = 512 #2048
    if inputs["input_ids"].shape[1] > MAX_HISTORY_TOKENS:
        for k in inputs:
            inputs[k] = inputs[k][:, -MAX_HISTORY_TOKENS:]

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    generated_tokens = outputs[0][input_len:]
    reply = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    if "<end_of_turn>" in reply:
        reply = reply.split("<end_of_turn>")[0].strip()

    return reply

## Gradio Interface

In [ ]:
import gradio as gr

gr.ChatInterface(
    fn=chat,
    title=f"Chat with {model_id}",
    description="Gradio running in Google Colab",
    examples=["Tell me a fact about Barcelona", "Explain mass timber in architecture", "How is creative image generation with AI changing architecture?", "Which architects are thought leaders for creative image generation in architecture?"],
    cache_examples=False # creates and caches responses to the examples above before loading
).launch(share=True, debug=True)


## Ask one question (without Gradio)

In [ ]:
# Example input
prompt = input("Hi! What can I help you with? ")

# Tokenize and send to device
inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096)
inputs = {k: v.to(model.device) for k, v in inputs.items() if isinstance(v, torch.Tensor)}

# Generate
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, temperature=0.7, top_p=0.9)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(response)